# DATA CLEANING AND FEATURE ENGINEERING

In [10]:
# main libraries used
import pandas as pd
import numpy as np
import os

In [12]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("maksbasher/ufc-complete-dataset-all-events-1996-2024")

print("Path to dataset files:", path)

# look at all data files
print(os.listdir(path))

# look at inner files from the folder
inner_path = os.path.join(path, "UFC dataset")
print(os.listdir(inner_path))

Path to dataset files: C:\Users\joaqu\.cache\kagglehub\datasets\maksbasher\ufc-complete-dataset-all-events-1996-2024\versions\6
['UFC dataset']
['Fighter stats', 'Large set', 'Medium set', 'Small set', 'Urls']


In [13]:
# Paths to the datasets that will be used for the project

base_path = path
large_set_path = os.path.join(base_path, "UFC dataset", "Large set")
fighter_stats_path = os.path.join(base_path, "UFC dataset", "Fighter stats")
medium_set_with_event_date = os.path.join(base_path, "UFC dataset", "Medium set")

In [14]:
# extract csv files from the Kaggle API

def load_csvs_from_folder(folder_path):
    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    dataframes = {}

    for file in csv_files:
        full_path = os.path.join(folder_path, file)
        df = pd.read_csv(full_path, encoding="utf-8", low_memory=False)
        dataframes[file] = df
        print(f"Loaded: {file} — shape {df.shape}")

    return df

df = load_csvs_from_folder(large_set_path)

Loaded: large_dataset.csv — shape (7439, 95)


In [15]:
# load second dataset with dates
url = "https://raw.githubusercontent.com/jhidalgo-05/UFC-Winner-Prediction/refs/heads/main/large_dataset_with_event_date_from_ufcstats.csv" # file found in github

df2 = pd.read_csv(url)

In [16]:
# first look into the data
print(df.info())
print(df2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7439 entries, 0 to 7438
Data columns (total 95 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   event_name              7439 non-null   object 
 1   r_fighter               7439 non-null   object 
 2   b_fighter               7439 non-null   object 
 3   winner                  7439 non-null   object 
 4   weight_class            7439 non-null   object 
 5   is_title_bout           7439 non-null   int64  
 6   gender                  7439 non-null   object 
 7   method                  7439 non-null   object 
 8   finish_round            7439 non-null   int64  
 9   total_rounds            7408 non-null   float64
 10  time_sec                7439 non-null   int64  
 11  referee                 7407 non-null   object 
 12  r_kd                    7439 non-null   int64  
 13  r_sig_str               7439 non-null   int64  
 14  r_sig_str_att           7439 non-null   

In [17]:
# load lookup table
date_lookup = df2[['event_name', 'r_fighter', 'b_fighter', 'event_date']]

# Ensure uniqueness (VERY IMPORTANT)
date_lookup = date_lookup.drop_duplicates(
    subset=['event_name', 'r_fighter', 'b_fighter']
)

# Remove old date columns
df = df.drop(columns=['event_date', 'event_date_x', 'event_date_y', 'date'], errors='ignore')

# Merge
df = df.merge(
    date_lookup,
    on=['event_name', 'r_fighter', 'b_fighter'],
    how='left'
)

# Validation checks
print("Missing event_date:", df['event_date'].isna().sum())
print("Duplicate rows:", df.duplicated().sum())

Missing event_date: 0
Duplicate rows: 0


In [18]:
# preserve fighter names before removing them
fighter_names = df[['r_fighter', 'b_fighter']]

### Gender-Based Model Separation

The dataset includes both men's and women's fights, which may exhibit different statistical patterns (e.g., striking rates, fight pace, and finishing tendencies).  

To account for these differences and improve model performance, we will split the dataset by gender and train separate models for each group. This allows each model to better capture the unique dynamics within men's and women's divisions, rather than forcing a single model to generalize across both.

In [19]:
# remove and store women fights for a separate study and accuracy purposes
wdf = df[df["gender"] == "Women"].copy()
mdf = df.drop(df[df["gender"] == "Women"].index).copy()

In [10]:
print(mdf.shape)
print(wdf.shape)

(6706, 96)
(733, 96)


In [10]:
# confirming different statistical patterns between genders
target = ['reach_diff', 'age_diff', 'height_diff', 'weight_diff', 'total_rounds']

for targets in target:
    print(f'Mean for {targets}:\n')
    print(np.mean(mdf[targets]))
    print(np.mean(wdf[targets]))
    print('####################\n')

Mean for reach_diff:

0.2077736906638638
0.0440462427745665
####################

Mean for age_diff:

0.39565686123517635
0.32060027285129605
####################

Mean for height_diff:

0.021968386519534415
0.2668212824010909
####################

Mean for weight_diff:

0.17752758723531162
0.11128240109140505
####################

Mean for total_rounds:

3.1208988764044943
3.2019099590723057
####################



In [20]:
# look for missing values
missing = mdf.isnull().sum()
missing = missing[missing > 0]

print(missing)

total_rounds     31
referee          27
r_age            76
r_reach         400
r_stance         24
b_age           190
b_reach         859
b_stance         63
age_diff        213
reach_diff      997
dtype: int64


In [21]:
# mean imputation for all missing values (numeric)
vars = ["total_rounds", "r_age", "r_reach", "b_age", "b_reach", "age_diff", "reach_diff"]
for var in vars:   
    mdf[var] = mdf[var].fillna(mdf[var].mean())

In [22]:
# mode imputation for all missing values (categorical)
vars = ["r_stance", "b_stance"]
for var in vars:   
    mdf[var] = mdf[var].fillna(mdf[var].mode()[0])
    
# double check
missing = mdf.isnull().sum()
missing = missing[missing > 0]

print(missing)

referee    27
dtype: int64


In [23]:
# check for duplicated fights
mdf.duplicated(subset=['r_fighter','b_fighter','event_date']).sum()

np.int64(0)

In [24]:
# check for class imbalance
mdf['winner'].value_counts(normalize=True)

winner
Red     0.663883
Blue    0.336117
Name: proportion, dtype: float64

In [25]:
df_balanced = mdf.copy()

### Addressing Class Imbalance

To prevent the model from being biased toward the majority class, we address class imbalance in the dataset. This ensures that the model learns meaningful patterns from both outcomes rather than favoring the more frequent class, leading to more reliable and generalizable predictions.

Invert r and b fighter statistics in a separate dataset. Concatenate and ensure 50/50 balance.

In [18]:
"""
df_balanced = mdf.copy()
flipped = df_balanced.copy()

# swap fighters + winner
flipped[['r_fighter', 'b_fighter']] = flipped[['b_fighter', 'r_fighter']].values
flipped['winner'] = flipped['winner'].map({'Red': 'Blue', 'Blue': 'Red'})

# only flip columns that have BOTH r_ and b_ versions
r_cols = [c for c in df_balanced.columns if c.startswith('r_')]

for r_col in r_cols:
    b_col = r_col.replace('r_', 'b_')

    # ONLY flip if both exist
    if b_col in df_balanced.columns:
        flipped[r_col], flipped[b_col] = df_balanced[b_col], df_balanced[r_col]

# combine datasets
df_balanced = pd.concat([df_balanced, flipped], ignore_index=True)

# check balance
print(df_balanced['winner'].value_counts(normalize=True))
"""

"\ndf_balanced = mdf.copy()\nflipped = df_balanced.copy()\n\n# swap fighters + winner\nflipped[['r_fighter', 'b_fighter']] = flipped[['b_fighter', 'r_fighter']].values\nflipped['winner'] = flipped['winner'].map({'Red': 'Blue', 'Blue': 'Red'})\n\n# only flip columns that have BOTH r_ and b_ versions\nr_cols = [c for c in df_balanced.columns if c.startswith('r_')]\n\nfor r_col in r_cols:\n    b_col = r_col.replace('r_', 'b_')\n\n    # ONLY flip if both exist\n    if b_col in df_balanced.columns:\n        flipped[r_col], flipped[b_col] = df_balanced[b_col], df_balanced[r_col]\n\n# combine datasets\ndf_balanced = pd.concat([df_balanced, flipped], ignore_index=True)\n\n# check balance\nprint(df_balanced['winner'].value_counts(normalize=True))\n"

Pre-Fight Feature Engineering for UFC Prediction Model
-----------------------------
This cell prepares the dataset for p
predicting UFC fight outcomes using only information
 available before the fight. It merges each fighter's historical career stats (e.g.,
 average strikes per minute, takedown accuracy, submissions) into the fight dataset,
 computes differences between Red and Blue fighters for each feature, and produces
 a clean feature matrix (X_men) and target vector (y_men) ready for modeling.

In [26]:
import re
from sklearn.preprocessing import LabelEncoder

# 1. Define the 8 core divisions as a search pattern
# We use a "|" (OR) to search for any of these keywords
core_pattern = 'Flyweight|Bantamweight|Featherweight|Lightweight|Welterweight|Middleweight|Light Heavyweight|Heavyweight'

# 2. Extract the core name from messy strings
# This converts "UFC Bantamweight Title" -> "Bantamweight"
# flags=re.IGNORECASE handles "lightweight" vs "Lightweight"
df_balanced['weight_class'] = df_balanced['weight_class'].str.extract(f'({core_pattern})', flags=re.IGNORECASE, expand=False)

# 3. Drop rows that didn't match any of the 8 (e.g., Strawweight, Catchweight, or NaNs)
df_balanced = df_balanced.dropna(subset=['weight_class']).copy()

# 4. Standardize capitalization
# .str.title() ensures "Light Heavyweight" is capitalized correctly
df_balanced['weight_class'] = df_balanced['weight_class'].str.title()

# --- Continue with feature matrix creation ---
pre_fight_features = [
    "weight_class", "age_diff", "height_diff", "weight_diff", "reach_diff",
    "wins_total_diff", "losses_total_diff", "r_stance", "b_stance"
]

X_men = df_balanced[pre_fight_features].copy()
y_men = df_balanced["winner"]

# One-Hot Encode and strip the prefix as discussed
X_men = pd.get_dummies(X_men, columns=['weight_class', 'r_stance', 'b_stance'], drop_first=True)
X_men.columns = [col.replace('weight_class_', '') for col in X_men.columns]

# Fill missing values
X_men = X_men.fillna(X_men.mean())

# label encode the target
le = LabelEncoder()
y_men = le.fit_transform(y_men)

# Sanity Check
#print(f"Cleaned Divisions: {X_men.columns[-7:].tolist()}") # Shows the last few columns (your weight classes)
#X_men.head(20)

print(df_balanced.head(20))

                              event_name              r_fighter  \
1   UFC Fight Night: Ribas vs. Namajunas          Karl Williams   
2   UFC Fight Night: Ribas vs. Namajunas       Edmen Shahbazyan   
3   UFC Fight Night: Ribas vs. Namajunas         Payton Talbott   
4   UFC Fight Night: Ribas vs. Namajunas      Billy Quarantillo   
5   UFC Fight Night: Ribas vs. Namajunas       Fernando Padilla   
6   UFC Fight Night: Ribas vs. Namajunas         Kurt Holobaugh   
7   UFC Fight Night: Ribas vs. Namajunas          Ricardo Ramos   
8   UFC Fight Night: Ribas vs. Namajunas            Miles Johns   
9   UFC Fight Night: Ribas vs. Namajunas           Jarno Errens   
11  UFC Fight Night: Ribas vs. Namajunas          Igor Severino   
12  UFC Fight Night: Ribas vs. Namajunas         Mohammed Usman   
13   UFC Fight Night: Tuivasa vs. Tybura            Tai Tuivasa   
14   UFC Fight Night: Tuivasa vs. Tybura     Ovince Saint Preux   
15   UFC Fight Night: Tuivasa vs. Tybura    Christian Rodrigue

In [27]:
df_balanced['weight_class'].value_counts()

weight_class
Lightweight          1281
Welterweight         1249
Middleweight          987
Featherweight         711
Heavyweight           683
Light Heavyweight     660
Bantamweight          633
Flyweight             324
Name: count, dtype: int64

In [28]:
# Create a new 'winner_name' column
# If 'winner' is 'Red', use the name in 'r_fighter'; otherwise, use the name in 'b_fighter'
df_balanced['winner_name'] = np.where(df_balanced['winner'] == 'Red', df_balanced['r_fighter'], df_balanced['b_fighter'])

# Verify the results
print("Recent fight outcomes with winner names:")
display(df_balanced[['r_fighter', 'b_fighter', 'winner', 'winner_name']].head())

Recent fight outcomes with winner names:


,r_fighter,b_fighter,winner,winner_name
1,Karl Williams,Justin Tafa,Red,Karl Williams
2,Edmen Shahbazyan,AJ Dobson,Red,Edmen Shahbazyan
3,Payton Talbott,Cameron Saaiman,Red,Payton Talbott
4,Billy Quarantillo,Youssef Zalal,Blue,Youssef Zalal
5,Fernando Padilla,Luis Pajuelo,Red,Fernando Padilla


## MODELING

Time to start modeling. We will use a simple SVM model to set up as a benchmark before continuing to engineer rolling features that can give our models an idea of how fighters are performing before the fight. This avoids target leakage risk seen in some of the variables present in the dataset.

In [21]:
"""
# SVM Benchmark Model

import random

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# set seed for reproducibility
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

# create SVM pipeline. avoid data leakage
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced', kernel='rbf', random_state=SEED))
])

# use standard 5 fold cross validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,   # randomly shuffle observations before creating folds
    random_state=SEED
)

# cross validated predictions
y_pred = cross_val_predict(
    svm_model,
    X_men,
    y_men,
    cv=cv
)

# print results
accuracy = accuracy_score(y_men, y_pred)
precision = precision_score(y_men, y_pred)
recall = recall_score(y_men, y_pred)
f1 = f1_score(y_men, y_pred)

print("SVM Benchmark (5-Fold CV)\n")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
"""

'\n# SVM Benchmark Model\n\nimport random\n\nfrom sklearn.svm import SVC\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import StratifiedKFold, cross_val_predict\nfrom sklearn.metrics import (\n    accuracy_score,\n    precision_score,\n    recall_score,\n    f1_score\n)\n\n# set seed for reproducibility\nSEED = 42\n\nnp.random.seed(SEED)\nrandom.seed(SEED)\n\n# create SVM pipeline. avoid data leakage\nsvm_model = Pipeline([\n    (\'scaler\', StandardScaler()),\n    (\'svm\', SVC(class_weight=\'balanced\', kernel=\'rbf\', random_state=SEED))\n])\n\n# use standard 5 fold cross validation\ncv = StratifiedKFold(\n    n_splits=5,\n    shuffle=True,   # randomly shuffle observations before creating folds\n    random_state=SEED\n)\n\n# cross validated predictions\ny_pred = cross_val_predict(\n    svm_model,\n    X_men,\n    y_men,\n    cv=cv\n)\n\n# print results\naccuracy = accuracy_score(y_men, y_pred)\nprecision = pre

In [22]:
"""
# hyperparameter tuning using GridSearch

from sklearn.model_selection import GridSearchCV

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto'],
    'svm__kernel': ['rbf'],
    'svm__class_weight': ['balanced']
}

grid = GridSearchCV(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC())
    ]),
    param_grid=param_grid,
    cv=cv,
    scoring='f1'
)

grid.fit(X_men, y_men)

print("BEST PARAMETERS:")
print(grid.best_params_)
print("\nBEST CV F1 SCORE:", grid.best_score_)
"""

'\n# hyperparameter tuning using GridSearch\n\nfrom sklearn.model_selection import GridSearchCV\n\nparam_grid = {\n    \'svm__C\': [0.1, 1, 10, 100],\n    \'svm__gamma\': [\'scale\', \'auto\'],\n    \'svm__kernel\': [\'rbf\'],\n    \'svm__class_weight\': [\'balanced\']\n}\n\ngrid = GridSearchCV(\n    estimator=Pipeline([\n        (\'scaler\', StandardScaler()),\n        (\'svm\', SVC())\n    ]),\n    param_grid=param_grid,\n    cv=cv,\n    scoring=\'f1\'\n)\n\ngrid.fit(X_men, y_men)\n\nprint("BEST PARAMETERS:")\nprint(grid.best_params_)\nprint("\nBEST CV F1 SCORE:", grid.best_score_)\n'

GridSearch was really slow. Will improve later on. Consider RandomizedSearch or n_jobs=-1 to use all available CPUs

In [23]:
"""
best_model = grid.best_estimator_

y_pred_best = cross_val_predict(
    best_model,
    X_men,
    y_men,
    cv=cv
)

print("TUNED SVM (5-Fold CV)\n")
print("Accuracy :", accuracy_score(y_men, y_pred_best))
print("Precision:", precision_score(y_men, y_pred_best))
print("Recall   :", recall_score(y_men, y_pred_best))
print("F1 Score :", f1_score(y_men, y_pred_best))
"""

'\nbest_model = grid.best_estimator_\n\ny_pred_best = cross_val_predict(\n    best_model,\n    X_men,\n    y_men,\n    cv=cv\n)\n\nprint("TUNED SVM (5-Fold CV)\n")\nprint("Accuracy :", accuracy_score(y_men, y_pred_best))\nprint("Precision:", precision_score(y_men, y_pred_best))\nprint("Recall   :", recall_score(y_men, y_pred_best))\nprint("F1 Score :", f1_score(y_men, y_pred_best))\n'

In [24]:
"""
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Compute confusion matrix
cm = confusion_matrix(y_men, y_pred_best)

# Print raw values
print("Confusion Matrix:")
print(cm)

# Plot
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Tuned SVM Confusion Matrix")
plt.show()
"""

'\nfrom sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay\nimport matplotlib.pyplot as plt\n\n# Compute confusion matrix\ncm = confusion_matrix(y_men, y_pred_best)\n\n# Print raw values\nprint("Confusion Matrix:")\nprint(cm)\n\n# Plot\ndisp = ConfusionMatrixDisplay(confusion_matrix=cm)\ndisp.plot()\n\nplt.title("Tuned SVM Confusion Matrix")\nplt.show()\n'

## separate approach

linear kernel

In [25]:
"""
# set seed for reproducibility
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

# create SVM pipeline. avoid data leakage
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced', kernel='linear', random_state=SEED))
])

# use standard 5 fold cross validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,   # randomly shuffle observations before creating folds
    random_state=SEED
)

# cross validated predictions
y_pred = cross_val_predict(
    svm_model,
    X_men,
    y_men,
    cv=cv
)

# print results
accuracy = accuracy_score(y_men, y_pred)
precision = precision_score(y_men, y_pred)
recall = recall_score(y_men, y_pred)
f1 = f1_score(y_men, y_pred)

print("SVM Benchmark (5-Fold CV)\n")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
"""

'\n# set seed for reproducibility\nSEED = 42\n\nnp.random.seed(SEED)\nrandom.seed(SEED)\n\n# create SVM pipeline. avoid data leakage\nsvm_model = Pipeline([\n    (\'scaler\', StandardScaler()),\n    (\'svm\', SVC(class_weight=\'balanced\', kernel=\'linear\', random_state=SEED))\n])\n\n# use standard 5 fold cross validation\ncv = StratifiedKFold(\n    n_splits=5,\n    shuffle=True,   # randomly shuffle observations before creating folds\n    random_state=SEED\n)\n\n# cross validated predictions\ny_pred = cross_val_predict(\n    svm_model,\n    X_men,\n    y_men,\n    cv=cv\n)\n\n# print results\naccuracy = accuracy_score(y_men, y_pred)\nprecision = precision_score(y_men, y_pred)\nrecall = recall_score(y_men, y_pred)\nf1 = f1_score(y_men, y_pred)\n\nprint("SVM Benchmark (5-Fold CV)\n")\nprint(f"Accuracy : {accuracy:.4f}")\nprint(f"Precision: {precision:.4f}")\nprint(f"Recall   : {recall:.4f}")\nprint(f"F1 Score : {f1:.4f}")\n'

In [26]:
"""
# hyperparameter tuning using GridSearch

from sklearn.model_selection import GridSearchCV

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto'],
    'svm__kernel': ['linear'],
    'svm__class_weight': ['balanced']
}

grid = GridSearchCV(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC())
    ]),
    param_grid=param_grid,
    cv=cv,
    scoring='f1'
)

grid.fit(X_men, y_men)

print("BEST PARAMETERS:")
print(grid.best_params_)
print("\nBEST CV F1 SCORE:", grid.best_score_)
"""

'\n# hyperparameter tuning using GridSearch\n\nfrom sklearn.model_selection import GridSearchCV\n\nparam_grid = {\n    \'svm__C\': [0.1, 1, 10, 100],\n    \'svm__gamma\': [\'scale\', \'auto\'],\n    \'svm__kernel\': [\'linear\'],\n    \'svm__class_weight\': [\'balanced\']\n}\n\ngrid = GridSearchCV(\n    estimator=Pipeline([\n        (\'scaler\', StandardScaler()),\n        (\'svm\', SVC())\n    ]),\n    param_grid=param_grid,\n    cv=cv,\n    scoring=\'f1\'\n)\n\ngrid.fit(X_men, y_men)\n\nprint("BEST PARAMETERS:")\nprint(grid.best_params_)\nprint("\nBEST CV F1 SCORE:", grid.best_score_)\n'

In [27]:
"""
best_model = grid.best_estimator_

y_pred_best = cross_val_predict(
    best_model,
    X_men,
    y_men,
    cv=cv
)

print("TUNED SVM (5-Fold CV)\n")
print("Accuracy :", accuracy_score(y_men, y_pred_best))
print("Precision:", precision_score(y_men, y_pred_best))
print("Recall   :", recall_score(y_men, y_pred_best))
print("F1 Score :", f1_score(y_men, y_pred_best))
"""

'\nbest_model = grid.best_estimator_\n\ny_pred_best = cross_val_predict(\n    best_model,\n    X_men,\n    y_men,\n    cv=cv\n)\n\nprint("TUNED SVM (5-Fold CV)\n")\nprint("Accuracy :", accuracy_score(y_men, y_pred_best))\nprint("Precision:", precision_score(y_men, y_pred_best))\nprint("Recall   :", recall_score(y_men, y_pred_best))\nprint("F1 Score :", f1_score(y_men, y_pred_best))\n'

In [28]:
"""
# Compute confusion matrix
cm = confusion_matrix(y_men, y_pred_best)

# Print raw values
print("Confusion Matrix:")
print(cm)

# Plot
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Tuned SVM Confusion Matrix")
plt.show()
"""

'\n# Compute confusion matrix\ncm = confusion_matrix(y_men, y_pred_best)\n\n# Print raw values\nprint("Confusion Matrix:")\nprint(cm)\n\n# Plot\ndisp = ConfusionMatrixDisplay(confusion_matrix=cm)\ndisp.plot()\n\nplt.title("Tuned SVM Confusion Matrix")\nplt.show()\n'

## Another Approach

polynomial kernel

In [29]:
"""
# set seed for reproducibility
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

# create SVM pipeline. avoid data leakage
svm_model = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(class_weight='balanced', kernel='poly', random_state=SEED))
])

# use standard 5 fold cross validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,   # randomly shuffle observations before creating folds
    random_state=SEED
)

# cross validated predictions
y_pred = cross_val_predict(
    svm_model,
    X_men,
    y_men,
    cv=cv
)

# print results
accuracy = accuracy_score(y_men, y_pred)
precision = precision_score(y_men, y_pred)
recall = recall_score(y_men, y_pred)
f1 = f1_score(y_men, y_pred)

print("SVM Benchmark (5-Fold CV)\n")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
"""

'\n# set seed for reproducibility\nSEED = 42\n\nnp.random.seed(SEED)\nrandom.seed(SEED)\n\n# create SVM pipeline. avoid data leakage\nsvm_model = Pipeline([\n    (\'scaler\', StandardScaler()),\n    (\'svm\', SVC(class_weight=\'balanced\', kernel=\'poly\', random_state=SEED))\n])\n\n# use standard 5 fold cross validation\ncv = StratifiedKFold(\n    n_splits=5,\n    shuffle=True,   # randomly shuffle observations before creating folds\n    random_state=SEED\n)\n\n# cross validated predictions\ny_pred = cross_val_predict(\n    svm_model,\n    X_men,\n    y_men,\n    cv=cv\n)\n\n# print results\naccuracy = accuracy_score(y_men, y_pred)\nprecision = precision_score(y_men, y_pred)\nrecall = recall_score(y_men, y_pred)\nf1 = f1_score(y_men, y_pred)\n\nprint("SVM Benchmark (5-Fold CV)\n")\nprint(f"Accuracy : {accuracy:.4f}")\nprint(f"Precision: {precision:.4f}")\nprint(f"Recall   : {recall:.4f}")\nprint(f"F1 Score : {f1:.4f}")\n'

In [30]:
"""
# hyperparameter tuning using GridSearch

from sklearn.model_selection import GridSearchCV

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto'],
    'svm__kernel': ['poly'],
    'svm__class_weight': ['balanced']
}

grid = GridSearchCV(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC())
    ]),
    param_grid=param_grid,
    cv=cv,
    scoring='f1'
)

grid.fit(X_men, y_men)

print("BEST PARAMETERS:")
print(grid.best_params_)
print("\nBEST CV F1 SCORE:", grid.best_score_)
"""

'\n# hyperparameter tuning using GridSearch\n\nfrom sklearn.model_selection import GridSearchCV\n\nparam_grid = {\n    \'svm__C\': [0.1, 1, 10, 100],\n    \'svm__gamma\': [\'scale\', \'auto\'],\n    \'svm__kernel\': [\'poly\'],\n    \'svm__class_weight\': [\'balanced\']\n}\n\ngrid = GridSearchCV(\n    estimator=Pipeline([\n        (\'scaler\', StandardScaler()),\n        (\'svm\', SVC())\n    ]),\n    param_grid=param_grid,\n    cv=cv,\n    scoring=\'f1\'\n)\n\ngrid.fit(X_men, y_men)\n\nprint("BEST PARAMETERS:")\nprint(grid.best_params_)\nprint("\nBEST CV F1 SCORE:", grid.best_score_)\n'

In [31]:
"""
best_model = grid.best_estimator_

y_pred_best = cross_val_predict(
    best_model,
    X_men,
    y_men,
    cv=cv
)

print("TUNED SVM (5-Fold CV)\n")
print("Accuracy :", accuracy_score(y_men, y_pred_best))
print("Precision:", precision_score(y_men, y_pred_best))
print("Recall   :", recall_score(y_men, y_pred_best))
print("F1 Score :", f1_score(y_men, y_pred_best))
"""

'\nbest_model = grid.best_estimator_\n\ny_pred_best = cross_val_predict(\n    best_model,\n    X_men,\n    y_men,\n    cv=cv\n)\n\nprint("TUNED SVM (5-Fold CV)\n")\nprint("Accuracy :", accuracy_score(y_men, y_pred_best))\nprint("Precision:", precision_score(y_men, y_pred_best))\nprint("Recall   :", recall_score(y_men, y_pred_best))\nprint("F1 Score :", f1_score(y_men, y_pred_best))\n'

In [32]:
"""
# Compute confusion matrix
cm = confusion_matrix(y_men, y_pred_best)

# Print raw values
print("Confusion Matrix:")
print(cm)

# Plot
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Tuned SVM Confusion Matrix")
plt.show()
"""

'\n# Compute confusion matrix\ncm = confusion_matrix(y_men, y_pred_best)\n\n# Print raw values\nprint("Confusion Matrix:")\nprint(cm)\n\n# Plot\ndisp = ConfusionMatrixDisplay(confusion_matrix=cm)\ndisp.plot()\n\nplt.title("Tuned SVM Confusion Matrix")\nplt.show()\n'

# Fighter Score

In [29]:
def exponential_recent_form(history, decay=0.75, recent_n=5):
    if not history:
        return 0.5

    recent_history = history[-recent_n:]

    outcomes = np.array(
        [fight["win"] for fight in recent_history],
        dtype=float
    )

    # Oldest fight gets the smallest weight,
    # newest fight gets weight 1
    ages = np.arange(len(outcomes) - 1, -1, -1)
    weights = decay ** ages

    return np.average(outcomes, weights=weights)

In [30]:
def calculate_simple_score(history, recent_n=5, decay=0.75):
    if len(history) == 0:
        return 50.0

    total_fights = len(history)
    total_wins = sum(fight["win"] for fight in history)

    career_win_rate = total_wins / total_fights

    recent_form = exponential_recent_form(
        history,
        decay=decay,
        recent_n=recent_n
    )

    finishes = sum(
        fight["win"] and fight["finish"]
        for fight in history
    )

    finish_rate = finishes / max(total_wins, 1)

    streak = 0

    for fight in reversed(history):
        if fight["win"]:
            if streak < 0:
                break
            streak += 1
        else:
            if streak > 0:
                break
            streak -= 1

    streak = np.clip(streak, -3, 3)

    experience_score = np.log1p(total_fights)

    score = (
        30 * career_win_rate
        + 30 * recent_form
        + 5 * streak
        + 10 * finish_rate
        + 5 * experience_score
    )

    return score

Sort fights chronologically and create the fighter histories

In [31]:
from collections import defaultdict

# Make sure event_date is datetime
df_balanced["event_date"] = pd.to_datetime(df_balanced["event_date"])

# Sort chronologically
df_balanced = df_balanced.sort_values("event_date").reset_index(drop=True)

# Dictionary that will store each fighter's PRIOR fight history
fighter_history = defaultdict(list)

r_scores = []
b_scores = []

for _, row in df_balanced.iterrows():

    red = row["r_fighter"]
    blue = row["b_fighter"]

    # IMPORTANT:
    # Calculate fighter scores BEFORE adding current fight result
    red_score = calculate_simple_score(fighter_history[red])
    blue_score = calculate_simple_score(fighter_history[blue])

    r_scores.append(red_score)
    b_scores.append(blue_score)

    # Determine outcome
    red_won = row["winner"] == "Red"
    blue_won = row["winner"] == "Blue"

    # Determine whether fight ended in a finish
    method = str(row["method"]).lower()

    finish = (
        "ko" in method
        or "tko" in method
        or "submission" in method
        or "sub" in method
    )

    # Add CURRENT fight to history only AFTER scores were calculated
    fighter_history[red].append({
        "win": int(red_won),
        "finish": int(finish and red_won)
    })

    fighter_history[blue].append({
        "win": int(blue_won),
        "finish": int(finish and blue_won)
    })


# Add scores back to dataset
df_balanced["r_fighter_score"] = r_scores
df_balanced["b_fighter_score"] = b_scores

# Difference is what we'll mainly give the model
df_balanced["fighter_score_diff"] = (
    df_balanced["r_fighter_score"]
    - df_balanced["b_fighter_score"]
)

Example: Jon Jones

In [33]:
fighter = "Jon Jones"

df_balanced[
    (df_balanced["r_fighter"] == fighter) |
    (df_balanced["b_fighter"] == fighter)
][
    [
        "event_date",
        "r_fighter",
        "b_fighter",
        "winner",
        "r_fighter_score",
        "b_fighter_score"
    ]
]

,event_date,r_fighter,b_fighter,winner,r_fighter_score,b_fighter_score
838,2008-08-09,Jon Jones,Andre Gusmao,Red,50.000000,50.000000
932,2009-01-31,Jon Jones,Stephan Bonnar,Red,68.465736,66.056225
1029,2009-07-11,Jon Jones,Jake O'Brien,Red,75.493061,54.323021
1111,2009-12-05,Matt Hamill,Jon Jones,Red,73.176691,85.264805
1170,2010-03-21,Brandon Vera,Jon Jones,Blue,49.676225,47.909094
1265,2010-08-01,Jon Jones,Vladimir Matyushenko,Red,65.583637,68.294409
1402,2011-02-05,Jon Jones,Ryan Bader,Red,75.198181,87.958797
1432,2011-03-19,Mauricio Rua,Jon Jones,Blue,61.472242,83.629633
1568,2011-09-24,Jon Jones,Quinton Jackson,Red,86.267584,73.600603
1651,2011-12-10,Jon Jones,Lyoto Machida,Red,90.679592,63.507902


In [34]:
# sanity check
f_score = "fighter_score_diff"

print(f_score in df_balanced)

True


In [35]:
df_balanced[['fighter_score_diff']].head(25)

,fighter_score_diff
0,0.000000
1,0.000000
2,0.000000
3,-28.465736
4,0.000000
5,-51.534264
6,28.465736
7,28.465736
8,0.000000
9,0.000000


Updating pre fight features

In [37]:
pre_fight_features = [
    "weight_class",
    "age_diff",
    "height_diff",
    "weight_diff",
    "reach_diff",
    "wins_total_diff",
    "losses_total_diff",
    "fighter_score_diff",
    "r_stance",
    "b_stance"
]

Rerun the rest of your existing preprocessing:

In [ ]:
"""
X_men = df_balanced[pre_fight_features].copy()
y_men = df_balanced["winner"]

# encode categorical variables
X_men = pd.get_dummies(
    X_men,
    columns=["weight_class", "r_stance", "b_stance"],
    drop_first=True
)

X_men.columns = [
    col.replace("weight_class_", "")
    for col in X_men.columns
]

# mean imputation of missing values
X_men = X_men.fillna(X_men.mean())

# encode target
le = LabelEncoder()
y_men = le.fit_transform(y_men)
"""

Running SVM with fighter score included

In [42]:
"""
import random

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

svm_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "svm",
        SVC(
            class_weight="balanced",
            kernel="rbf",
            random_state=SEED
        )
    )
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

y_pred = cross_val_predict(
    svm_model,
    X_men,
    y_men,
    cv=cv
)

accuracy = accuracy_score(y_men, y_pred)
precision = precision_score(y_men, y_pred)
recall = recall_score(y_men, y_pred)
f1 = f1_score(y_men, y_pred)

print("SVM + Fighter Score (5-Fold CV)\n")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
"""

'\nimport random\n\nfrom sklearn.svm import SVC\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.model_selection import StratifiedKFold, cross_val_predict\nfrom sklearn.metrics import (\n    accuracy_score,\n    precision_score,\n    recall_score,\n    f1_score\n)\n\nSEED = 42\n\nnp.random.seed(SEED)\nrandom.seed(SEED)\n\nsvm_model = Pipeline([\n    ("scaler", StandardScaler()),\n    (\n        "svm",\n        SVC(\n            class_weight="balanced",\n            kernel="rbf",\n            random_state=SEED\n        )\n    )\n])\n\ncv = StratifiedKFold(\n    n_splits=5,\n    shuffle=True,\n    random_state=SEED\n)\n\ny_pred = cross_val_predict(\n    svm_model,\n    X_men,\n    y_men,\n    cv=cv\n)\n\naccuracy = accuracy_score(y_men, y_pred)\nprecision = precision_score(y_men, y_pred)\nrecall = recall_score(y_men, y_pred)\nf1 = f1_score(y_men, y_pred)\n\nprint("SVM + Fighter Score (5-Fold CV)\n")\nprint(f"Accuracy : {accuracy:.4f}"

## XGBOOST Model

In [43]:
#!pip install xgboost

In [44]:
"""
# import libraries (clean later)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
"""

'\n# import libraries (clean later)\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nfrom xgboost import XGBClassifier\n\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import OneHotEncoder, LabelEncoder\n\nfrom sklearn.model_selection import train_test_split, GridSearchCV\n\nfrom sklearn.metrics import (\n    accuracy_score,\n    classification_report,\n    confusion_matrix,\n    roc_auc_score\n)\n'

In [45]:
"""
# train/test/split of data
X_train_men, X_test_men, y_train_men, y_test_men = train_test_split(
    X_men,
    y_men,
    test_size=0.20,
    random_state=42,
    stratify=y_men
)

# calculate balance class weight

# XGBoost does not use class_weight="balanced".
# scale_pos_weight provides the equivalent weighting
# for binary classification
negative_count = np.sum(y_train_men == 0)
positive_count = np.sum(y_train_men == 1)

scale_pos_weight_men = negative_count / positive_count

print(f"scale_pos_weight: {scale_pos_weight_men:.3f}")
"""

'\n# train/test/split of data\nX_train_men, X_test_men, y_train_men, y_test_men = train_test_split(\n    X_men,\n    y_men,\n    test_size=0.20,\n    random_state=42,\n    stratify=y_men\n)\n\n# calculate balance class weight\n\n# XGBoost does not use class_weight="balanced".\n# scale_pos_weight provides the equivalent weighting\n# for binary classification\nnegative_count = np.sum(y_train_men == 0)\npositive_count = np.sum(y_train_men == 1)\n\nscale_pos_weight_men = negative_count / positive_count\n\nprint(f"scale_pos_weight: {scale_pos_weight_men:.3f}")\n'

In [46]:
"""
# build pipeline and fit model
xgb_pipeline_men = Pipeline(
    steps=[
        (
            "xgb",
            XGBClassifier(
                random_state=42,
                eval_metric="logloss",
                scale_pos_weight=scale_pos_weight_men
            )
        )
    ]
)

xgb_pipeline_men.set_params(
    xgb__n_estimators=200,
    xgb__max_depth=5,
    xgb__learning_rate=0.05,
    xgb__subsample=0.8
)

xgb_pipeline_men.fit(
    X_train_men,
    y_train_men
)
"""

'\n# build pipeline and fit model\nxgb_pipeline_men = Pipeline(\n    steps=[\n        (\n            "xgb",\n            XGBClassifier(\n                random_state=42,\n                eval_metric="logloss",\n                scale_pos_weight=scale_pos_weight_men\n            )\n        )\n    ]\n)\n\nxgb_pipeline_men.set_params(\n    xgb__n_estimators=200,\n    xgb__max_depth=5,\n    xgb__learning_rate=0.05,\n    xgb__subsample=0.8\n)\n\nxgb_pipeline_men.fit(\n    X_train_men,\n    y_train_men\n)\n'

In [47]:
"""
# evaluate model
y_pred_men = xgb_pipeline_men.predict(X_test_men)

y_prob_men = (
    xgb_pipeline_men
    .predict_proba(X_test_men)[:, 1]
)

print("\nBaseline XGBoost Performance")
print("-" * 40)

print(
    f"Accuracy: "
    f"{accuracy_score(y_test_men, y_pred_men):.4f}"
)

print(
    f"ROC-AUC: "
    f"{roc_auc_score(y_test_men, y_prob_men):.4f}"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_men,
        y_pred_men,
        target_names=le.classes_
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_men,
        y_pred_men
    )
)
"""

'\n# evaluate model\ny_pred_men = xgb_pipeline_men.predict(X_test_men)\n\ny_prob_men = (\n    xgb_pipeline_men\n    .predict_proba(X_test_men)[:, 1]\n)\n\nprint("\nBaseline XGBoost Performance")\nprint("-" * 40)\n\nprint(\n    f"Accuracy: "\n    f"{accuracy_score(y_test_men, y_pred_men):.4f}"\n)\n\nprint(\n    f"ROC-AUC: "\n    f"{roc_auc_score(y_test_men, y_prob_men):.4f}"\n)\n\nprint("\nClassification Report:")\nprint(\n    classification_report(\n        y_test_men,\n        y_pred_men,\n        target_names=le.classes_\n    )\n)\n\nprint("\nConfusion Matrix:")\nprint(\n    confusion_matrix(\n        y_test_men,\n        y_pred_men\n    )\n)\n'

In [48]:
"""
# feature importance analysis (top 15)
fitted_xgb_men = (
    xgb_pipeline_men
    .named_steps["model"]
)

importance_men = pd.DataFrame({
    "Feature": X_train_men.columns,
    "Importance": fitted_xgb_men.feature_importances_
})

importance_men = importance_men.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 15 Features:")
print(
    importance_men
    .head(15)
    .to_string(index=False)
)

top_features_men = importance_men.head(15)

plt.figure(figsize=(10, 7))

plt.barh(
    top_features_men["Feature"][::-1],
    top_features_men["Importance"][::-1]
)

plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 XGBoost Feature Importances — Men's Fights")

plt.tight_layout()
plt.show()
"""

'\n# feature importance analysis (top 15)\nfitted_xgb_men = (\n    xgb_pipeline_men\n    .named_steps["model"]\n)\n\nimportance_men = pd.DataFrame({\n    "Feature": X_train_men.columns,\n    "Importance": fitted_xgb_men.feature_importances_\n})\n\nimportance_men = importance_men.sort_values(\n    by="Importance",\n    ascending=False\n)\n\nprint("\nTop 15 Features:")\nprint(\n    importance_men\n    .head(15)\n    .to_string(index=False)\n)\n\ntop_features_men = importance_men.head(15)\n\nplt.figure(figsize=(10, 7))\n\nplt.barh(\n    top_features_men["Feature"][::-1],\n    top_features_men["Importance"][::-1]\n)\n\nplt.xlabel("Importance")\nplt.ylabel("Feature")\nplt.title("Top 15 XGBoost Feature Importances — Men\'s Fights")\n\nplt.tight_layout()\nplt.show()\n'

In [49]:
"""
# Hyperparameter tuning
param_grid_men = {
    "xgb__n_estimators": [100, 200, 300],
    "xgb__max_depth": [3, 5, 7],
    "xgb__learning_rate": [0.05, 0.10],
    "xgb__subsample": [0.8, 1.0]
}

# use gridsearch
grid_search_men = GridSearchCV(
    estimator=xgb_pipeline_men,
    param_grid=param_grid_men,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)


grid_search_men.fit(
    X_train_men,
    y_train_men
)
"""

'\n# Hyperparameter tuning\nparam_grid_men = {\n    "xgb__n_estimators": [100, 200, 300],\n    "xgb__max_depth": [3, 5, 7],\n    "xgb__learning_rate": [0.05, 0.10],\n    "xgb__subsample": [0.8, 1.0]\n}\n\n# use gridsearch\ngrid_search_men = GridSearchCV(\n    estimator=xgb_pipeline_men,\n    param_grid=param_grid_men,\n    scoring="roc_auc",\n    cv=5,\n    n_jobs=-1,\n    verbose=1\n)\n\n\ngrid_search_men.fit(\n    X_train_men,\n    y_train_men\n)\n'

In [50]:
"""
# best hyperparameters
print("Best Hyperparameters")

print(grid_search_men.best_params_)

print(
    f"\nBest CV ROC-AUC: "
    f"{grid_search_men.best_score_:.4f}"
)
"""

'\n# best hyperparameters\nprint("Best Hyperparameters")\n\nprint(grid_search_men.best_params_)\n\nprint(\n    f"\nBest CV ROC-AUC: "\n    f"{grid_search_men.best_score_:.4f}"\n)\n'

In [51]:
"""
# evaluate tuned model

best_xgb_men = grid_search_men.best_estimator_


y_pred_tuned_men = best_xgb_men.predict(
    X_test_men
)

y_prob_tuned_men = (
    best_xgb_men
    .predict_proba(X_test_men)[:, 1]
)


tuned_accuracy_men = accuracy_score(
    y_test_men,
    y_pred_tuned_men
)

tuned_auc_men = roc_auc_score(
    y_test_men,
    y_prob_tuned_men
)


print("\n" + "=" * 55)
print("TUNED XGBOOST")
print("=" * 55)

print(f"Accuracy: {tuned_accuracy_men:.4f}")
print(f"ROC-AUC:  {tuned_auc_men:.4f}")

print("\nClassification Report:")

print(
    classification_report(
        y_test_men,
        y_pred_tuned_men,
        target_names=le.classes_
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test_men,
        y_pred_tuned_men
    )
)
"""

'\n# evaluate tuned model\n\nbest_xgb_men = grid_search_men.best_estimator_\n\n\ny_pred_tuned_men = best_xgb_men.predict(\n    X_test_men\n)\n\ny_prob_tuned_men = (\n    best_xgb_men\n    .predict_proba(X_test_men)[:, 1]\n)\n\n\ntuned_accuracy_men = accuracy_score(\n    y_test_men,\n    y_pred_tuned_men\n)\n\ntuned_auc_men = roc_auc_score(\n    y_test_men,\n    y_prob_tuned_men\n)\n\n\nprint("\n" + "=" * 55)\nprint("TUNED XGBOOST")\nprint("=" * 55)\n\nprint(f"Accuracy: {tuned_accuracy_men:.4f}")\nprint(f"ROC-AUC:  {tuned_auc_men:.4f}")\n\nprint("\nClassification Report:")\n\nprint(\n    classification_report(\n        y_test_men,\n        y_pred_tuned_men,\n        target_names=le.classes_\n    )\n)\n\nprint("Confusion Matrix:")\n\nprint(\n    confusion_matrix(\n        y_test_men,\n        y_pred_tuned_men\n    )\n)\n'

FEATURE ENGINEERING (Rolling Statistics)

In [38]:
from collections import defaultdict

# 1. MAKE SURE FIGHTS ARE CHRONOLOGICAL

df_balanced["event_date"] = pd.to_datetime(
    df_balanced["event_date"]
)

df_balanced = (
    df_balanced
    .sort_values("event_date")
    .reset_index(drop=True)
)

# 2. ROLLING VARIABLES

rolling_features = [
    "str_acc_diff",
    "td_acc_diff",
    "sub_att_diff",
    "SLpM_total_diff",
    "str_def_total_diff",
    "td_def_total_diff"
]

# 3. STORE EACH FIGHTER'S HISTORICAL DIFFERENTIAL VALUES

fighter_history = defaultdict(
    lambda: {
        feature: []
        for feature in rolling_features
    }
)

# 4. ROLLING FUNCTION

def rolling_previous_3(values):
    """
    Average the fighter's previous 3 fights.

    0 previous fights -> 0
    1 previous fight  -> average of 1
    2 previous fights -> average of 2
    3+ previous fights -> average of most recent 3
    """

    if len(values) == 0:
        return 0.0

    return np.mean(values[-3:])

# 5. CREATE STORAGE FOR PRE-FIGHT ROLLING VALUES

# Red corner
r_roll3 = {
    feature: []
    for feature in rolling_features
}


# Blue corner
b_roll3 = {
    feature: []
    for feature in rolling_features
}

# 6. PROCESS FIGHTS CHRONOLOGICALLY

for _, row in df_balanced.iterrows():

    red = row["r_fighter"]
    blue = row["b_fighter"]

    # STEP 1:
    # CALCULATE ROLLING AVERAGES BEFORE CURRENT FIGHT

    for feature in rolling_features:

        # Red fighter

        r_roll3[feature].append(
            rolling_previous_3(
                fighter_history[red][feature]
            )
        )

        # Blue fighter

        b_roll3[feature].append(
            rolling_previous_3(
                fighter_history[blue][feature]
            )
        )

    # STEP 2:
    # ADD CURRENT FIGHT TO HISTORY
    # AFTER CALCULATING ROLLING VALUES

    for feature in rolling_features:

        diff = row[feature]


        # Only add valid values
        if pd.notna(diff):

            # Red perspective

            fighter_history[red][feature].append(
                diff
            )

            # Blue perspective

            fighter_history[blue][feature].append(
                -diff
            )

# 7. ADD PRE-FIGHT ROLLING VALUES TO DATAFRAME

for feature in rolling_features:

    df_balanced[f"r_{feature}_roll3"] = (
        r_roll3[feature]
    )

    df_balanced[f"b_{feature}_roll3"] = (
        b_roll3[feature]
    )

In [40]:
# RED / BLUE ROLLING FEATURES

df_balanced["r_Str_acc_roll3"] = r_roll3["str_acc_diff"]
df_balanced["b_Str_acc_roll3"] = b_roll3["str_acc_diff"]

df_balanced["r_Td_acc_roll3"] = r_roll3["td_acc_diff"]
df_balanced["b_Td_acc_roll3"] = b_roll3["td_acc_diff"]

df_balanced["r_Sub_att_roll3"] = r_roll3["sub_att_diff"]
df_balanced["b_Sub_att_roll3"] = b_roll3["sub_att_diff"]

df_balanced["r_SLpM_total_roll3"] = r_roll3["SLpM_total_diff"]
df_balanced["b_SLpM_total_roll3"] = b_roll3["SLpM_total_diff"]

df_balanced["r_Str_def_total_roll3"] = r_roll3["str_def_total_diff"]
df_balanced["b_Str_def_total_roll3"] = b_roll3["str_def_total_diff"]

df_balanced["r_Td_def_total_roll3"] = r_roll3["td_def_total_diff"]
df_balanced["b_Td_def_total_roll3"] = b_roll3["td_def_total_diff"]

# RED - BLUE ROLLING DIFFERENCES

df_balanced["Str_acc_roll3_diff"] = (
    df_balanced["r_Str_acc_roll3"]
    - df_balanced["b_Str_acc_roll3"]
)

df_balanced["Td_acc_roll3_diff"] = (
    df_balanced["r_Td_acc_roll3"]
    - df_balanced["b_Td_acc_roll3"]
)

df_balanced["Sub_att_roll3_diff"] = (
    df_balanced["r_Sub_att_roll3"]
    - df_balanced["b_Sub_att_roll3"]
)

df_balanced["SLpM_total_roll3_diff"] = (
    df_balanced["r_SLpM_total_roll3"]
    - df_balanced["b_SLpM_total_roll3"]
)

df_balanced["Str_def_total_roll3_diff"] = (
    df_balanced["r_Str_def_total_roll3"]
    - df_balanced["b_Str_def_total_roll3"]
)

df_balanced["Td_def_total_roll3_diff"] = (
    df_balanced["r_Td_def_total_roll3"]
    - df_balanced["b_Td_def_total_roll3"]
)

Perform a sanity check to see if rolling variables were actually created

In [41]:
# New rolling metafeatures
rolling_features = [
    "Str_acc_roll3_diff",
    "Td_acc_roll3_diff",
    "Sub_att_roll3_diff",
    "SLpM_total_roll3_diff",
    "Str_def_total_roll3_diff",
    "Td_def_total_roll3_diff"
]

# Check that columns exist
print("Columns found:")
for col in rolling_features:
    print(f"{col}: {col in df_balanced.columns}")

# Summary statistics
print("\nSummary statistics:")
display(
    df_balanced[rolling_features].describe()
)

# Missing values
print("\nMissing values:")
print(
    df_balanced[rolling_features]
    .isna()
    .sum()
)

Columns found:
Str_acc_roll3_diff: True
Td_acc_roll3_diff: True
Sub_att_roll3_diff: True
SLpM_total_roll3_diff: True
Str_def_total_roll3_diff: True
Td_def_total_roll3_diff: True

Summary statistics:


,Str_acc_roll3_diff,Td_acc_roll3_diff,Sub_att_roll3_diff,SLpM_total_roll3_diff,Str_def_total_roll3_diff,Td_def_total_roll3_diff
count,6528.000000,6528.000000,6528.000000,6528.000000,6528.000000,6528.000000
mean,0.008678,0.025779,0.017310,0.064141,0.010143,0.022001
std,0.216004,0.474513,1.125694,1.616513,0.116019,0.320629
min,-1.350000,-2.000000,-8.000000,-8.015000,-0.660000,-1.250000
25%,-0.115000,-0.266667,-0.500000,-0.896667,-0.053333,-0.160000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.126667,0.320000,0.500000,1.040000,0.073333,0.210000
max,1.156667,2.000000,10.333333,7.483333,0.590000,1.360000



Missing values:
Str_acc_roll3_diff          0
Td_acc_roll3_diff           0
Sub_att_roll3_diff          0
SLpM_total_roll3_diff       0
Str_def_total_roll3_diff    0
Td_def_total_roll3_diff     0
dtype: int64


In [42]:
# set rolling dataframe features up
rolling_features = [
    "Str_acc_roll3_diff",
    "Td_acc_roll3_diff",
    "Sub_att_roll3_diff",
    "SLpM_total_roll3_diff",
    "Str_def_total_roll3_diff",
    "Td_def_total_roll3_diff"
]

In [43]:
df_balanced["weight_class"].value_counts()

weight_class
Lightweight          1281
Welterweight         1249
Middleweight          987
Featherweight         711
Heavyweight           683
Light Heavyweight     660
Bantamweight          633
Flyweight             324
Name: count, dtype: int64

In [44]:
# final dataframe before modeling
final_df = pd.concat(
    [df_balanced[pre_fight_features], df_balanced[rolling_features]],
    axis=1)

# Save to Downloads folder
downloads_path = os.path.join(
    os.path.expanduser("~"),
    "Downloads",
    "final_fight_features.csv"
)

final_df.to_csv(downloads_path, index=False)

print(f"Saved successfully to: {downloads_path}")

Saved successfully to: C:\Users\joaqu\Downloads\final_fight_features.csv
